# Driver Drowsiness Detection — Week 7
### Real-Time Pipeline — Component Testing
NTCC | Amity School of Engineering & Technology | June 2026

Both models are trained and saved. This notebook checks that the pieces of the real-time pipeline actually work on real images before wiring everything together in the live webcam script (src/realtime_detection.py, which runs locally since Colab has no camera access).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

DRIVE  = '/content/drive/MyDrive/NTCC_Drowsiness_Project'
TRAIN  = f'{DRIVE}/data/train'
RES    = f'{DRIVE}/results'
MODELS = f'{DRIVE}/models'

print('TF:', tf.__version__)

## Loading both trained models

In [ ]:
eye_model  = load_model(f'{MODELS}/eye_model.h5')
face_model = load_model(f'{MODELS}/face_model.h5')
print('Both models loaded OK')

## Sanity check — run both models on real dataset images

Before wiring this into a live camera loop, confirming the models actually predict correctly on images pulled straight from the dataset.

In [ ]:
IMG_SIZE = 96

for cls, expected in [('Closed', 0), ('Open', 1)]:
    folder = os.path.join(TRAIN, cls)
    imgs   = os.listdir(folder)[:3]
    for fname in imgs:
        img = cv2.imread(os.path.join(folder, fname))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        inp = preprocess_input(np.expand_dims(img.astype('float32'), 0))
        pred = eye_model.predict(inp, verbose=0)[0]
        label = 'Closed' if np.argmax(pred) == 0 else 'Open'
        conf  = pred[np.argmax(pred)] * 100
        status = 'correct' if np.argmax(pred) == expected else 'WRONG'
        print(f'  [{status}] {cls} image -> predicted {label} ({conf:.1f}%)')

In [ ]:
for cls, expected in [('yawn', 0), ('no_yawn', 1)]:
    folder = os.path.join(TRAIN, cls)
    imgs   = os.listdir(folder)[:3]
    for fname in imgs:
        img = cv2.imread(os.path.join(folder, fname))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        inp = preprocess_input(np.expand_dims(img.astype('float32'), 0))
        pred = face_model.predict(inp, verbose=0)[0]
        label = 'yawn' if np.argmax(pred) == 0 else 'no_yawn'
        conf  = pred[np.argmax(pred)] * 100
        status = 'correct' if np.argmax(pred) == expected else 'WRONG'
        print(f'  [{status}] {cls} image -> predicted {label} ({conf:.1f}%)')

## EAR / MAR functions

Same functions that will run inside the live loop — testing them here on dummy coordinates first.

In [ ]:
from scipy.spatial import distance as dist

def calculate_EAR(eye_pts):
    A = dist.euclidean(eye_pts[1], eye_pts[5])
    B = dist.euclidean(eye_pts[2], eye_pts[4])
    C = dist.euclidean(eye_pts[0], eye_pts[3])
    return round((A + B) / (2.0 * C), 4)

def calculate_MAR(mouth_pts):
    A = dist.euclidean(mouth_pts[1], mouth_pts[7])
    B = dist.euclidean(mouth_pts[2], mouth_pts[6])
    C = dist.euclidean(mouth_pts[3], mouth_pts[5])
    D = dist.euclidean(mouth_pts[0], mouth_pts[4])
    return round((A + B + C) / (3.0 * D), 4)

open_eye   = [(0,0),(1,2),(2,2),(4,0),(3,2),(1,2)]
closed_eye = [(0,0),(1,0.2),(2,0.2),(4,0),(3,0.2),(1,0.2)]
print('EAR open eye   :', calculate_EAR(open_eye))
print('EAR closed eye :', calculate_EAR(closed_eye))

closed_mouth = [(0,0),(1,0.3),(2,0.3),(3,0.3),(6,0),(5,0.3),(4,0.3),(3,0.3)]
open_mouth   = [(0,0),(1,1.5),(2,1.5),(3,1.5),(6,0),(5,1.5),(4,1.5),(3,1.5)]
print('MAR closed mouth:', calculate_MAR(closed_mouth))
print('MAR open mouth  :', calculate_MAR(open_mouth))

## Drowsiness score fusion function

In [ ]:
def drowsiness_score(ear, mar, tilt, eye_closed_prob, yawn_prob,
                      ear_thresh=0.25, mar_thresh=0.50, tilt_thresh=20):
    ear_sig  = max(0, (ear_thresh - ear) / ear_thresh)
    mar_sig  = max(0, (mar - mar_thresh) / (1 - mar_thresh))
    tilt_sig = max(0, (tilt - tilt_thresh) / (90 - tilt_thresh))
    score = (0.30*eye_closed_prob + 0.30*yawn_prob + 0.20*ear_sig + 0.10*mar_sig + 0.10*tilt_sig)
    return round(min(score, 1.0), 4)

print('Fully awake  :', drowsiness_score(0.30, 0.25, 5, eye_closed_prob=0.05, yawn_prob=0.05))
print('Eyes closing :', drowsiness_score(0.15, 0.25, 5, eye_closed_prob=0.85, yawn_prob=0.05))
print('Yawning      :', drowsiness_score(0.28, 0.70, 5, eye_closed_prob=0.10, yawn_prob=0.80))
print('Fully drowsy :', drowsiness_score(0.10, 0.80, 40, eye_closed_prob=0.90, yawn_prob=0.85))

## Note on MediaPipe testing

MediaPipe's Face Mesh needs a live camera feed to be tested properly, and Colab doesn't have access to one. The actual landmark detection, camera capture, and full real-time loop are implemented and tested in `src/realtime_detection.py`, which runs locally on a laptop with a webcam. This notebook only verifies the pieces that don't need a camera — the trained models and the geometry functions — before they get wired into that script.

In [ ]:
print('='*50)
print('WEEK 7 SUMMARY')
print('='*50)
print('Eye model inference   : verified on real dataset images')
print('Yawn model inference  : verified on real dataset images')
print('EAR / MAR functions   : verified on test coordinates')
print('Drowsiness score      : verified across 4 test scenarios')
print()
print('Next: wire all of this into src/realtime_detection.py')
print('and test with an actual webcam locally.')
print('='*50)